In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/synthetic_covid_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Shape: (2400, 155)

First 5 rows:


,patient_id,age,sex,fever,cough,dyspnea,fatigue,headache,hypertension,diabetes,...,wavelet_014,wavelet_015,wavelet_016,wavelet_017,wavelet_018,wavelet_019,wavelet_020,ICU,mechanical_ventilation,death
0,P0001,57,0,0,1,0,1,0,0,0,...,-0.972052,-2.550070,-0.311566,0.868858,1.444569,-0.022637,0.180994,0,0,0
1,P0002,34,0,1,0,0,1,1,0,0,...,-0.875412,1.650275,1.353153,-0.679142,1.987041,0.120426,1.393416,0,0,0
2,P0003,65,0,1,1,0,0,0,1,1,...,0.509871,1.569078,0.068379,0.942279,0.389688,2.734910,-0.042336,1,1,1
3,P0004,68,0,0,1,0,0,1,0,0,...,-0.246149,-0.958745,0.140452,-1.069833,-0.288854,0.370459,0.482073,0,1,0
4,P0005,19,0,1,0,0,0,0,0,0,...,-1.255056,-1.511882,-1.616099,-0.182825,-0.631720,-2.478670,-0.721141,0,0,0


In [3]:
clinical_features = [
    "age",
    "sex",
    "hypertension",
    "diabetes",
    "cardiovascular_disease",
    "chronic_lung_disease",
    "immunosuppression",
    "smoking_history",
    "dyspnea",
    "fever",
    "cough",
    "respiratory_rate",
    "oxygen_saturation",
    "systolic_bp",
    "heart_rate"
]

lab_features = [
    "wbc",
    "lymphocyte",
    "neutrophil",
    "hemoglobin",
    "platelet",
    "crp",
    "ldh",
    "d_dimer",
    "ferritin",
    "ast",
    "alt",
    "creatinine",
    "urea",
    "albumin",
    "procalcitonin",
    "troponin",
    "il6"
]

target_features = [
    "ICU",
    "mechanical_ventilation",
    "death"
]

radiomics_features = [
    col for col in df.columns
    if col not in clinical_features + lab_features + target_features
]

print("Clinical features:", len(clinical_features))
print("Laboratory features:", len(lab_features))
print("Radiomics features:", len(radiomics_features))
print("Target features:", len(target_features))

print("\nTotal:",
      len(clinical_features) +
      len(lab_features) +
      len(radiomics_features) +
      len(target_features))

Clinical features: 15
Laboratory features: 17
Radiomics features: 132
Target features: 3

Total: 167


In [4]:
missing = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (missing / len(df)) * 100
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values("Missing_Percentage", ascending=False)

print("Columns containing missing values:")
display(missing_summary)

Columns containing missing values:


,Missing_Count,Missing_Percentage
d_dimer,138,5.750000
ldh,113,4.708333
crp,90,3.750000
creatinine,86,3.583333
albumin,74,3.083333
...,...,...
glrlm_007,15,0.625000
wavelet_003,15,0.625000
glrlm_003,14,0.583333
glrlm_015,14,0.583333


In [5]:
print("Duplicate rows:", df.duplicated().sum())

if "patient_id" in df.columns:
    print("Duplicate patient IDs:", df["patient_id"].duplicated().sum())

Duplicate rows: 0
Duplicate patient IDs: 0


In [6]:
print(df.dtypes.value_counts())

display(
    pd.DataFrame({
        "Feature": df.columns,
        "Data_Type": df.dtypes.astype(str)
    }).head(30)
)

float64    137
int64       17
object       1
Name: count, dtype: int64


,Feature,Data_Type
patient_id,patient_id,object
age,age,int64
sex,sex,int64
fever,fever,int64
cough,cough,int64
dyspnea,dyspnea,int64
fatigue,fatigue,int64
headache,headache,int64
hypertension,hypertension,int64
diabetes,diabetes,int64


In [7]:
for target in target_features:
    counts = df[target].value_counts().sort_index()
    percentages = df[target].value_counts(normalize=True).sort_index() * 100

    print(f"\n{target}")
    print("-" * 40)

    for value in counts.index:
        print(
            f"{value}: {counts[value]} patients "
            f"({percentages[value]:.2f}%)"
        )


ICU
----------------------------------------
0: 1696 patients (70.67%)
1: 704 patients (29.33%)

mechanical_ventilation
----------------------------------------
0: 1730 patients (72.08%)
1: 670 patients (27.92%)

death
----------------------------------------
0: 2009 patients (83.71%)
1: 391 patients (16.29%)


In [8]:
outcome_corr = df[target_features].corr()

print("Outcome correlation:")
display(outcome_corr)

Outcome correlation:


,ICU,mechanical_ventilation,death
ICU,1.000000,0.309007,0.216362
mechanical_ventilation,0.309007,1.000000,0.246102
death,0.216362,0.246102,1.000000


In [10]:
# Let's find any column names that match our clinical numeric variables
possible_keywords = ['rate', 'sat', 'bp', 'oxygen', 'heart', 'resp', 'age']
matching_cols = [col for col in df.columns if any(kw in col.lower() for kw in possible_keywords)]
print("Matching columns found in dataset:", matching_cols)

# Let's describe the clinical numeric columns that are actually present
clinical_numeric = [col for col in ['age', 'respiratory_rate', 'oxygen_saturation', 'systolic_bp', 'heart_rate'] if col in df.columns]

if clinical_numeric:
    display(df[clinical_numeric].describe().T)
else:
    print("\nNone of the expected numeric clinical columns were found. Here are the first 15 columns available:")
    print(df.columns[:15].tolist())

Matching columns found in dataset: ['age']


,count,mean,std,min,25%,50%,75%,max
age,2400.0,51.382083,16.463364,18.0,40.0,52.0,62.0,90.0


In [13]:
# Filter lab_features to keep only the columns that actually exist in the DataFrame
existing_lab_features = [col for col in lab_features if col in df.columns]

# Describe the available laboratory features
display(df[existing_lab_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
wbc,2348.0,7.488064,2.026678,2.000000,6.140795,7.491012,8.876825,15.078217
lymphocyte,2349.0,1.559196,0.581925,0.200000,1.167446,1.550980,1.955860,3.577046
neutrophil,2400.0,4.742773,1.805065,0.500000,3.534889,4.773297,5.921899,12.046355
hemoglobin,2400.0,13.792379,1.403328,9.028652,12.824778,13.771639,14.729656,18.000000
platelet,2400.0,229.520626,46.373366,51.118449,199.961192,229.657946,259.336703,411.209642
crp,2310.0,51.229125,35.291918,0.000000,22.638807,50.264661,74.764562,172.865621
ldh,2287.0,329.301748,128.936160,100.000000,233.127632,324.147456,417.702829,848.523082
d_dimer,2262.0,2.016214,0.632881,0.096751,1.582133,1.993721,2.429007,4.513453
ast,2400.0,30.603922,15.273660,5.000000,19.566351,30.090144,40.476074,83.350174
alt,2400.0,28.945622,13.867088,5.000000,19.246700,28.530018,38.308982,83.115050


In [14]:
display(df[radiomics_features].describe().T.head(20))

,count,mean,std,min,25%,50%,75%,max
fatigue,2400.0,0.481667,0.499768,0.000000,0.000000,0.000000,1.000000,1.000000
headache,2400.0,0.256667,0.436885,0.000000,0.000000,0.000000,1.000000,1.000000
copd,2400.0,0.063333,0.243612,0.000000,0.000000,0.000000,0.000000,1.000000
chronic_liver_disease,2400.0,0.057917,0.233634,0.000000,0.000000,0.000000,0.000000,1.000000
chronic_kidney_disease,2400.0,0.072083,0.258680,0.000000,0.000000,0.000000,0.000000,1.000000
cancer,2400.0,0.047083,0.211861,0.000000,0.000000,0.000000,0.000000,1.000000
bilirubin,2400.0,13.104818,4.552112,3.000000,9.848722,13.165888,16.182622,27.554043
potassium,2400.0,4.141879,0.364555,2.844708,3.898319,4.147166,4.388912,5.324678
sodium,2400.0,137.866948,2.554015,129.795220,136.124007,137.807348,139.614611,146.918381
ck,2400.0,125.611049,57.683094,20.000000,85.133190,126.049384,164.785296,344.361955


In [15]:
validation_report = {
    "Number of patients": len(df),
    "Number of features": len(df.columns),
    "Clinical features": len(clinical_features),
    "Laboratory features": len(lab_features),
    "Radiomics features": len(radiomics_features),
    "Target variables": len(target_features),
    "Duplicate rows": int(df.duplicated().sum()),
    "Total missing cells": int(df.isnull().sum().sum()),
    "Missing percentage": round(
        df.isnull().sum().sum() / df.size * 100, 3
    ),
    "ICU positive": int(df["ICU"].sum()),
    "Mechanical ventilation positive": int(
        df["mechanical_ventilation"].sum()
    ),
    "Death positive": int(df["death"].sum())
}

validation_df = pd.DataFrame(
    validation_report.items(),
    columns=["Metric", "Value"]
)

display(validation_df)

,Metric,Value
0,Number of patients,2400.000
1,Number of features,155.000
2,Clinical features,15.000
3,Laboratory features,17.000
4,Radiomics features,132.000
5,Target variables,3.000
6,Duplicate rows,0.000
7,Total missing cells,3456.000
8,Missing percentage,0.929
9,ICU positive,704.000
